In [1]:
import os


import pandas as pd
import matplotlib as plt


import pandas as pd
import matplotlib.pyplot as plt

df_clean = pd.read_csv("HealthConnect_clean.csv")

In [2]:
total_appointments = len(df_clean)

no_shows = (
    df_clean["appointment_outcome"] == "No-Show"
).sum()

no_show_rate = (
    no_shows / total_appointments
) * 100

total_appointments, no_shows, no_show_rate

(5000, np.int64(2423), np.float64(48.46))

### Test: Overall No-Show KPI validation
### Expected: 5,000 appointments, 2,423 no-shows, 48.46%
### Actual: 5,000, 2,423, 48.46%
### Result: ✅ Passed
### Issue: None identified
### Action: No refinement required

In [3]:
long_lead = df_clean[
    df_clean["lead_time_group"] == "45-60 days"
]

long_lead_appointments = len(long_lead)

long_lead_no_shows = (
    long_lead["appointment_outcome"] == "No-Show"
).sum()

long_lead_rate = (
    long_lead_no_shows / long_lead_appointments
) * 100

long_lead_appointments, long_lead_no_shows, long_lead_rate

(1251, np.int64(841), np.float64(67.22621902478018))

### Test: 45–60 day No-Show Rate validation
### Expected: 1,251 appointments, 841 no-shows, 67.23%
### Actual: 1,251, 841, 67.2262%
### Result: ✅ Passed
### Issue: None identified
### Action: No refinement required

In [4]:
high_risk = df_clean[
    (df_clean["lead_time_group"] == "45-60 days") &
    (df_clean["distance_group"] == "15+ km")
]

high_risk_appointments = len(high_risk)

high_risk_no_shows = (
    high_risk["appointment_outcome"] == "No-Show"
).sum()

high_risk_rate = (
    high_risk_no_shows / high_risk_appointments
) * 100

high_risk_appointments, high_risk_no_shows, high_risk_rate

(237, np.int64(173), np.float64(72.9957805907173))

### Test: High-risk segment validation
### Expected: 237 appointments, 173 no-shows, 73.00%
### Actual: 237, 173, 72.9958%
### Result: ✅ Passed
### Issue: None identified
### Action: No refinement required

In [5]:
high_risk_difference = high_risk_rate - no_show_rate

high_risk_difference

np.float64(24.5357805907173)

### Test: High-risk difference validation
### Expected: +24.54 pp
### Actual: +24.5358 pp → +24.54 pp
### Result: ✅ Passed
### Issue: None identified
### Action: No refinement required

In [6]:
previous_no_show_validation = (
    df_clean
    .groupby("previous_no_shows")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

previous_no_show_validation

previous_no_shows
0     43.51
1     53.49
2     59.36
3     67.95
4     66.67
5    100.00
Name: appointment_outcome, dtype: float64

### Test: Previous no-show behaviour validation
### Expected: Week 6 rates reproduced
### Actual: Exact match
### Result: ✅ Passed
### Issue: Sparse higher-history groups
### Action: Keep 0–3 groups as the main evidence; avoid over-interpreting 4–5.

In [7]:
distance_validation = (
    df_clean
    .groupby("distance_group")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

distance_validation

distance_group
0-5 km      46.45
10-15 km    48.50
15+ km      54.09
5-10 km     46.51
Name: appointment_outcome, dtype: float64

### The results match the Week 6 analysis exactly:

### 0–5 km → 46.45%
### 5–10 km → 46.51%
### 10–15 km → 48.50%
### 15+ km → 54.09%

In [9]:
reminder_validation = (
    df_clean
    .groupby("reminder_sent")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)
reminder_validation

reminder_sent
No     51.39
Yes    47.36
Name: appointment_outcome, dtype: float64

### No reminder: 51.39%
### Reminder sent: 47.36%
### Difference: 4.03 percentage points

## Testing the statistical findings
#### 1. Is booking lead time is still statistically associated with appointment outcome.

In [10]:
from scipy.stats import chi2_contingency

lead_time_test = pd.crosstab(
    df_clean["lead_time_group"],
    df_clean["appointment_outcome"]
)

chi2, p_value, dof, expected = chi2_contingency(lead_time_test)

chi2, p_value

(np.float64(381.7345150548457), np.float64(2.3571478127301484e-79))

### Expected: Significant association between booking lead time and appointment outcome.
### Actual: χ² = 381.7345, p = 2.36 × 10⁻⁷⁹.
### Issue identified: The exact χ² value differs from the Week 6 documented value because the Week 7 test uses the full three-category appointment outcome.
### Refinement: Record the contingency-table definition alongside the statistical result.
### Decision: Validated. The underlying conclusion is reliable.

### 2. We'll now test whether distance to clinic is significantly associated with appointment outcome, using the same three-outcome approach.

In [12]:
distance_test = pd.crosstab(
    df_clean["distance_group"],
    df_clean["appointment_outcome"]
)

chi2, p_value, dof, expected = chi2_contingency(distance_test)

chi2, p_value

(np.float64(18.732063911344945), np.float64(0.004640898388077408))

### Expected: Distance should show a statistically significant association with appointment outcome.
### Actual: χ² = 18.7321, p = 0.00464.
### Issue: Exact statistic differs from Week 6 because of the contingency-table definition.
### Refinement: Document the outcome categories used in statistical tests.
### Decision: Validated. Distance remains a statistically supported risk indicator.

## 3. previous no-show behaviour

In [13]:
previous_no_show_test = pd.crosstab(
    df_clean["previous_no_shows"],
    df_clean["appointment_outcome"]
)

chi2, p_value, dof, expected = chi2_contingency(
    previous_no_show_test
)

chi2, p_value

(np.float64(83.3757153469355), np.float64(1.0906187436066216e-13))

### Expected: Previous no-show behaviour should show a statistically significant association with appointment outcome.
### Actual: χ² = 83.3757, p = 1.09 × 10⁻¹³.
### Issue: The exact statistic differs slightly from the Week 6 result because this test uses the full three-category outcome.
### Refinement: Keep the statistical-test definition explicit and avoid presenting the χ² value without its contingency-table structure.
### Decision: Validated. Previous no-show behaviour remains a statistically supported risk indicator.

In [14]:
# Validate the high-risk segment definition
high_risk_check = df_clean[
    (df_clean["lead_time_group"] == "45-60 days") &
    (df_clean["distance_group"] == "15+ km")
]

high_risk_check[
    ["lead_time_group", "distance_group"]
].drop_duplicates()

,lead_time_group,distance_group
7,45-60 days,15+ km


In [15]:
# final check on the high-risk segment counts.
high_risk_check.shape[0], (
    high_risk_check["appointment_outcome"] == "No-Show"
).sum()

(237, np.int64(173))

In [16]:
# Validate the high-risk no-show rate
high_risk_rate_check = (
    (high_risk_check["appointment_outcome"] == "No-Show").sum()
    / high_risk_check.shape[0]
) * 100

high_risk_rate_check

np.float64(72.9957805907173)

In [17]:
# Validate the high-risk difference
high_risk_difference_check = high_risk_rate_check - no_show_rate
high_risk_difference_check

np.float64(24.5357805907173)

In [18]:
# Validate the business-impact calculation
expected_no_shows = long_lead_appointments * (no_show_rate / 100)

excess_no_shows = long_lead_no_shows - expected_no_shows

expected_no_shows, excess_no_shows

(np.float64(606.2346), np.float64(234.7654))

In [19]:
# Validate the 45–60 day group's contribution to all no-shows
long_lead_no_show_share = (
    long_lead_no_shows / no_shows
) * 100

long_lead_no_show_share

np.float64(34.70903838217087)

In [20]:
# Validate the high-risk segment's share of all no-shows
high_risk_no_show_share = (
    high_risk_no_shows / no_shows
) * 100

high_risk_no_show_share

np.float64(7.139909203466777)

In [21]:
# Validate the Week 5 distance finding
distance_15_plus = df_clean[
    df_clean["distance_group"] == "15+ km"
]

distance_15_plus.shape[0], (
    distance_15_plus["appointment_outcome"] == "No-Show"
).sum()

(930, np.int64(503))

In [22]:
# Validate the 15+ km no-show rate
distance_15_plus_rate = (
    503 / 930
) * 100

distance_15_plus_rate

54.086021505376344

In [23]:
# validate the 45–60 day booking lead-time rate of 67.23% using the underlying records one more time
long_lead_direct = df_clean[
    (df_clean["booking_lead_days"] >= 45) &
    (df_clean["booking_lead_days"] <= 60)
]

long_lead_direct.shape[0], (
    long_lead_direct["appointment_outcome"] == "No-Show"
).sum()

(1251, np.int64(841))

### 45–60 day appointments: 1,251 ✅
### No-shows: 841 ✅
### Matches the grouped calculation exactly ✅
### This is particularly valuable because it confirms that the lead-time grouping logic did not distort the strongest Week 5 finding.

In [24]:
# Validate the remaining lead-time groups
pd.crosstab(
    df_clean["lead_time_group"],
    pd.cut(
        df_clean["booking_lead_days"],
        bins=[-1, 14, 29, 44, 60],
        labels=["0-14 days", "15-29 days", "30-44 days", "45-60 days"]
    )
)

booking_lead_days,0-14 days,15-29 days,30-44 days,45-60 days
lead_time_group,,,,
0-14 days,1242,0,0,0
15-29 days,0,1250,0,0
30-44 days,0,0,1257,0
45-60 days,0,0,0,1251


### There are no records appearing in the wrong group. This confirms that lead_time_group correctly represents the underlying booking_lead_days values.

In [25]:
# Validate distance-group boundaries
pd.crosstab(
    df_clean["distance_group"],
    pd.cut(
        df_clean["distance_to_clinic_km"],
        bins=[0, 5, 10, 15, float("inf")],
        labels=["0-5 km", "5-10 km", "10-15 km", "15+ km"]
    )
)

distance_to_clinic_km,0-5 km,5-10 km,10-15 km,15+ km
distance_group,,,,
0-5 km,1156,0,0,0
10-15 km,0,0,1132,0
15+ km,0,0,0,930
5-10 km,0,1692,0,0


### There are no records crossing into the wrong distance group. So the distance_group transformation is correctly representing the raw distance_to_clinic_km values.

In [26]:
# Validate the reminder transformation
pd.crosstab(
    df_clean["reminder_sent"],
    df_clean["reminder_channel_model"]
)

reminder_channel_model,Email,No Reminder,SMS,WhatsApp
reminder_sent,,,,
No,0,1366,0,0
Yes,533,0,2000,1101


### reminder_sent = No → 1,366 No Reminder ✅
### reminder_sent = Yes → 533 Email, 2,000 SMS, 1,101 WhatsApp ✅
### No contradictory combinations exist ✅
### No missing values remain in reminder_channel_model ✅

### This confirms that our refinement for the structural missing reminder_channel values was implemented correctly.

In [27]:
# Validate the reminder-channel counts
df_clean["reminder_channel_model"].value_counts()

reminder_channel_model
SMS            2000
No Reminder    1366
WhatsApp       1101
Email           533
Name: count, dtype: int64

### SMS: 2,000 ✅
### No Reminder: 1,366 ✅
### WhatsApp: 1,101 ✅
### Email: 533 ✅

### Total = 5,000 records, so the transformation accounts for the entire dataset.

### At this point, we've validated both the logic and counts behind the reminder refinement.

In [28]:
# Validate the previous no-show grouping
df_clean["previous_no_shows"].value_counts().sort_index()

previous_no_shows
0    2921
1    1548
2     438
3      78
4      12
5       3
Name: count, dtype: int64

### Total = 5,000 records ✅

### This confirms why we need to be careful with the higher categories: 4 previous no-shows has only 12 records, and 5 has only 3 records. The strongest reliable evidence is therefore concentrated in the 0–3 previous no-show groups.

In [29]:
# Validate the previous no-show rates directly
previous_no_show_direct = (
    df_clean
    .groupby("previous_no_shows")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

previous_no_show_direct

previous_no_shows
0     43.51
1     53.49
2     59.36
3     67.95
4     66.67
5    100.00
Name: appointment_outcome, dtype: float64

### 0 previous no-shows: 43.51% — strong sample size
### 1: 53.49%
### 2: 59.36%
### 3: 67.95%
### 4: 66.67% — only 12 records
### 5: 100.00% — only 3 records

### The important pattern remains: no-show rates generally increase as previous no-show behaviour increases, particularly across the well-supported 0–3 groups.

In [30]:
# Validate the previous no-show finding for Data Science
(df_clean["previous_no_shows"] > df_clean["previous_appointments"]).sum()

np.int64(0)

### This confirms there are no records where previous no-shows exceed previous appointments. That gives us confidence that previous_no_shows is internally consistent and suitable for consideration as a Data Science model feature.

In [31]:
# Validate the Data Science target
pd.crosstab(
    df_clean["no_show_target"],
    df_clean["appointment_outcome"]
)

appointment_outcome,Attended,Cancelled,No-Show
no_show_target,,,
0,2314,263,0
1,0,0,2423


### No Attended/Cancelled records were incorrectly labelled 1 ✅
### No No-Show records were incorrectly labelled 0 ✅

### This is important because it confirms the target variable is clean and there is no obvious target-labeling error.

In [32]:
# Check for target leakage
candidate_features = [
    "booking_lead_days",
    "previous_appointments",
    "previous_no_shows",
    "distance_to_clinic_km",
    "reminder_sent",
    "reminder_channel_model",
    "appointment_type",
    "age_group",
    "appointment_time",
    "appointment_day"
]

"appointment_outcome" in candidate_features

False

### This is because using appointment_outcome as a predictor would leak the answer into the model.

In [33]:
# Validate missing-distance records
df_clean["distance_to_clinic_km"].isna().sum()

np.int64(90)

### This confirms the 90 missing distance values we identified during data-quality testing are still present and have not been accidentally removed or altered.

In [34]:
# Check whether missing distance affects the outcome
distance_missing_comparison = (
    df_clean
    .assign(
        distance_status=df_clean["distance_to_clinic_km"]
        .isna()
        .map({True: "Missing", False: "Available"})
    )
    .groupby("distance_status")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

distance_missing_comparison

distance_status
Available    48.39
Missing      52.22
Name: appointment_outcome, dtype: float64

#### The difference is noticeable but not large enough to justify removing the 90 records with missing distance. Keeping them preserves the dataset, while the missing distance values should be handled explicitly during any future modelling.

In [35]:
waiting_missing_comparison = (
    df_clean
    .assign(
        waiting_status=df_clean["waiting_time_minutes"]
        .isna()
        .map({True: "Missing", False: "Available"})
    )
    .groupby("waiting_status")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

waiting_missing_comparison

waiting_status
Available    48.30
Missing      61.67
Name: appointment_outcome, dtype: float64

#### This is a meaningful difference, so we should not treat the missing waiting-time records as completely neutral. However, we also should not delete them based on this comparison alone. The 60 missing records represent only 1.2% of the dataset.

In [36]:
# Check how many waiting-time values are missing
df_clean["waiting_time_minutes"].isna().sum()

np.int64(60)

In [38]:
# Check the Sunday business-rule inconsistency
sunday_appointments = (
    df_clean["appointment_day"] == "Sunday"
).sum()

sunday_appointments

np.int64(737)

### Result: ISSUE IDENTIFIED ⚠️

### The dataset contains 737 Sunday appointments.

### However, the stated HealthConnect business rule says the clinics are closed on Sundays. That means 14.74% of the 5,000 appointments occur on a day when the clinics are supposedly closed.

### This is a significant inconsistency.

In [39]:
# Check the Sunday no-show rate
sunday_no_show_rate = (
    df_clean.loc[
        df_clean["appointment_day"] == "Sunday",
        "appointment_outcome"
    ]
    .eq("No-Show")
    .mean() * 100
)

sunday_no_show_rate

np.float64(50.47489823609227)

In [40]:
# Check whether Sunday appointments affect the overall KPI
non_sunday = df_clean[df_clean["appointment_day"] != "Sunday"]

non_sunday_no_show_rate = (
    non_sunday["appointment_outcome"]
    .eq("No-Show")
    .mean() * 100
)

non_sunday.shape[0], non_sunday_no_show_rate

(4263, np.float64(48.11165845648605))

### So although the 737 Sunday appointments conflict with the stated business rule, they have very little impact on the overall no-show KPI.

In [41]:
# Verify the final dataset still has 5,000 records
df_clean.shape

(5000, 24)

In [42]:
# Check for duplicate appointments again
df_clean["appointment_id"].duplicated().sum()

np.int64(0)

### This confirms that our transformations did not create duplicate appointments or unintentionally replicate records.

In [43]:
# Check for missing values in the final dataset
df_clean.isna().sum()

appointment_id                   0
patient_id                       0
gender                           0
age                              0
age_group                        0
appointment_type                 0
booking_date                     0
appointment_date                 0
appointment_day                  0
appointment_time                 0
booking_lead_days                0
previous_appointments            0
previous_no_shows                0
reminder_sent                    0
reminder_channel              1366
distance_to_clinic_km           90
waiting_time_minutes            60
appointment_outcome              0
lead_time_group                  0
previous_appointment_group       0
distance_group                  90
waiting_time_group              60
no_show_target                   0
reminder_channel_model           0
dtype: int64

### No unexpected missingness was discovered. The remaining missing values are understood and documented rather than blindly removed.

In [45]:
# Final outcome reconciliation
outcome_counts = df_clean["appointment_outcome"].value_counts()

outcome_counts, outcome_counts.sum()

(appointment_outcome
 No-Show      2423
 Attended     2314
 Cancelled     263
 Name: count, dtype: int64,
 np.int64(5000))

### This confirms that no appointment records are unaccounted for or assigned to an unexpected outcome category.

# Reassess Week 6 Recommendation: Booking Lead Time

In [46]:
lead_time_type_validation = (
    df_clean
    .groupby(["appointment_type", "lead_time_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

lead_time_type_validation

appointment_type         lead_time_group
Diagnostic Test          0-14 days          30.88
                         15-29 days         43.70
                         30-44 days         51.83
                         45-60 days         68.99
Follow-up                0-14 days          32.97
                         15-29 days         43.64
                         30-44 days         57.34
                         45-60 days         70.57
General Consultation     0-14 days          29.25
                         15-29 days         42.75
                         30-44 days         49.81
                         45-60 days         64.84
Specialist Consultation  0-14 days          29.57
                         15-29 days         42.49
                         30-44 days         53.36
                         45-60 days         65.89
Name: appointment_outcome, dtype: float64

#### The booking lead-time pattern remains consistent across all four appointment types.

#### Every appointment type shows a clear increase from the 0–14 day group to the 45–60 day group:

#### Diagnostic Test: 30.88% → 68.99%
#### Follow-up: 32.97% → 70.57%
#### General Consultation: 29.25% → 64.84%
#### Specialist Consultation: 29.57% → 65.89%

#### The highest rate is 70.57% for Follow-up appointments booked 45–60 days ahead, while even the lowest 45–60 day rate is 64.84%.

In [47]:
# Reassess the Distance Recommendation
distance_lead_validation = (
    df_clean
    .groupby(["lead_time_group", "distance_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

distance_lead_validation

lead_time_group  distance_group
0-14 days        0-5 km            25.77
                 10-15 km          32.73
                 15+ km            36.78
                 5-10 km           28.61
15-29 days       0-5 km            38.28
                 10-15 km          42.96
                 15+ km            50.22
                 5-10 km           42.24
30-44 days       0-5 km            56.39
                 10-15 km          51.32
                 15+ km            56.82
                 5-10 km           49.33
45-60 days       0-5 km            66.22
                 10-15 km          66.55
                 15+ km            73.00
                 5-10 km           65.38
Name: appointment_outcome, dtype: float64

#### The distance pattern remains meaningful, especially for longer booking lead times.

#### For the 45–60 day group:

#### 0–5 km → 66.22%
#### 5–10 km → 65.38%
#### 10–15 km → 66.55%
#### 15+ km → 73.00%

#### The 15+ km group is clearly the highest at 73.00%.

#### More importantly, the 45–60 day booking lead time remains the dominant pattern across every distance group, while distance adds additional risk at the longest lead time.

In [48]:
# Reassess the reminder recommendation
reminder_lead_validation = (
    df_clean
    .groupby(["lead_time_group", "reminder_sent"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

reminder_lead_validation

lead_time_group  reminder_sent
0-14 days        No               32.32
                 Yes              29.98
15-29 days       No               46.46
                 Yes              41.69
30-44 days       No               53.74
                 Yes              52.46
45-60 days       No               73.46
                 Yes              65.05
Name: appointment_outcome, dtype: float64

#### The pattern is consistent across every lead-time group:

#### 0–14 days: No reminder 32.32% vs Yes 29.98% → 2.34 pp lower
#### 15–29 days: 46.46% vs 41.69% → 4.77 pp lower
#### 30–44 days: 53.74% vs 52.46% → 1.28 pp lower
#### 45–60 days: 73.46% vs 65.05% → 8.41 pp lower
What this validates

#### This supports the Week 6 recommendation that reminders should be considered as part of a targeted engagement strategy, particularly for appointments booked 45–60 days in advance.

#### The strongest observed difference is in the highest-risk lead-time group:

#### 73.46% → 65.05% = 8.41 percentage-point difference.

#### However, we must keep the interpretation careful: this is an observed association, not proof that reminders cause the reduction. Patients who receive reminders may differ from those who do not in other ways.

In [49]:
# Previous no-shows across booking lead-time groups
previous_no_show_lead_validation = (
    df_clean
    .groupby(["lead_time_group", "previous_no_shows"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

previous_no_show_lead_validation

lead_time_group  previous_no_shows
0-14 days        0                     26.74
                 1                     32.56
                 2                     47.54
                 3                     31.82
                 4                     50.00
                 5                    100.00
15-29 days       0                     37.94
                 1                     47.37
                 2                     50.85
                 3                     85.00
                 4                     75.00
30-44 days       0                     48.45
                 1                     58.25
                 2                     62.11
                 3                     66.67
                 4                     33.33
                 5                    100.00
45-60 days       0                     61.30
                 1                     72.89
                 2                     80.58
                 3                     90.48
                 4  

#### The previous-no-show pattern generally becomes stronger as booking lead time increases. The 4 and 5 previous-no-show groups are too sparse to rely on heavily, so we should not use their 100% values as major evidence.
#### This strengthens the Week 6 finding that previous no-show behaviour is an important risk indicator, and it shows that previous behaviour remains relevant even after considering booking lead time.

#### The combination is particularly concerning:

#### 45–60 day booking lead time + previous no-show history = very high observed no-show rates.

In [50]:
# Validate the primary recommendation
lead_time_final_validation = (
    df_clean
    .groupby("lead_time_group")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

lead_time_final_validation

lead_time_group
0-14 days     30.60
15-29 days    43.04
30-44 days    52.82
45-60 days    67.23
Name: appointment_outcome, dtype: float64

#### The no-show rate rises consistently as booking lead time increases:

#### 0–14 days: 30.60%
#### 15–29 days: 43.04%
#### 30–44 days: 52.82%
#### 45–60 days: 67.23%

#### The difference between the lowest and highest groups is:

#### 67.23% − 30.60% = 36.63 percentage points.

#### This is also consistent with all our earlier cross-segment testing. The 45–60 day group remains the clearest and strongest risk segment.
#### HealthConnect should prioritize proactive support for appointments booked 45–60 days in advance, because this group has the highest observed no-show risk.

In [52]:
# Final validation of the high-risk segment
high_risk_final_validation = {
    "High-risk appointments": len(high_risk),
    "High-risk no-shows": (
        high_risk["appointment_outcome"] == "No-Show"
    ).sum(),
    "High-risk no-show rate": round(
        (high_risk["appointment_outcome"] == "No-Show").mean() * 100,
        2
    ),
    "Overall no-show rate": round(no_show_rate, 2),
    "Difference (pp)": round(
        (
            (high_risk["appointment_outcome"] == "No-Show").mean() * 100
        ) - no_show_rate,
        2
    )
}

high_risk_final_validation

{'High-risk appointments': 237,
 'High-risk no-shows': np.int64(173),
 'High-risk no-show rate': np.float64(73.0),
 'Overall no-show rate': np.float64(48.46),
 'Difference (pp)': np.float64(24.54)}

#### The results confirm:

#### High-risk appointments: 237
#### High-risk no-shows: 173
#### High-risk no-show rate: 73.00%
#### Overall no-show rate: 48.46%
#### Difference: +24.54 percentage points

#### So this segment has a no-show rate substantially above the overall clinic level.
#### HealthConnect should give additional attention to appointments that are:

#### 45–60 days in advance + 15+ km from the clinic.

#### This segment should be treated as a priority risk group, not as a guaranteed no-show group.

In [53]:
# Final validation of previous no-show history
previous_no_show_final_validation = (
    df_clean
    .groupby("previous_no_shows")["appointment_outcome"]
    .agg(
        appointments="count",
        no_shows=lambda x: (x == "No-Show").sum(),
        no_show_rate=lambda x: (x == "No-Show").mean() * 100
    )
    .round(2)
)

previous_no_show_final_validation

,appointments,no_shows,no_show_rate
previous_no_shows,,,
0,2921,1271,43.51
1,1548,828,53.49
2,438,260,59.36
3,78,53,67.95
4,12,8,66.67
5,3,3,100.00


#### There is a strong overall relationship between previous no-shows and the current no-show rate:

#### 0 previous no-shows: 43.51% — 2,921 appointments
#### 1: 53.49% — 1,548 appointments
#### 2: 59.36% — 438 appointments
#### 3: 67.95% — 78 appointments
#### 4: 66.67% — only 12 appointments
#### 5: 100.00% — only 3 appointments

#### The main evidence is therefore the 0–3 previous no-show groups, because they have enough observations to support a more reliable conclusion.
#### HealthConnect should use previous no-show history as a risk indicator when deciding which patients may benefit from additional appointment support.

In [55]:
# Validate Data Science candidate features
data_science_feature_validation = {
    "booking_lead_days": df_clean["booking_lead_days"].notna().all(),
    "previous_no_shows": df_clean["previous_no_shows"].notna().all(),
    "distance_to_clinic_km": df_clean["distance_to_clinic_km"].notna().any(),
    "reminder_sent": df_clean["reminder_sent"].notna().all(),
    "reminder_channel_model": df_clean["reminder_channel_model"].notna().all(),
    "no_show_target": df_clean["no_show_target"].notna().all(),
    "appointment_outcome_in_features": "appointment_outcome" in candidate_features
}

data_science_feature_validation

{'booking_lead_days': np.True_,
 'previous_no_shows': np.True_,
 'distance_to_clinic_km': np.True_,
 'reminder_sent': np.True_,
 'reminder_channel_model': np.True_,
 'no_show_target': np.True_,
 'appointment_outcome_in_features': False}